In [ ]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("kde4", lang1="en", lang2="vi", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

kde4.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
raw_datasets["train"][1]["translation"]

{'en': 'Add Feeds to Akregator', 'vi': 'Thêm các nguồn tin cho Akregator'}

In [ ]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 38503
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 4279
    })
})

Đổi "test" thành "validation":

In [ ]:
split_datasets["validation"] = split_datasets["test"]

In [ ]:
split_datasets["train"][1]["translation"]

{'en': 'Document Contents', 'vi': 'Nội dung Tài liệu'}

In [ ]:
from transformers import pipeline

model_checkpoint = "Helsinki-NLP/opus-mt-vi-en"
translator = pipeline("translation", model=model_checkpoint)
translator("Thêm các nguồn tin cho Akregator")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


[{'translation_text': 'Add Feeds to Akregator'}]

In [ ]:
split_datasets["train"][4544]["translation"]

{'en': 'Inactive Titlebar', 'vi': 'Thanh tiêu đề không chọn'}

In [ ]:
translator("Thanh tiêu đề không chọn")

[{'translation_text': 'Unknown Titlebar'}]

## **Processing the data**

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-vi-en"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="tf")

In [ ]:
max_length = 128

def preprocess_function(examples):
  inputs = [ex["vi"] for ex in examples["translation"]]
  targets = [ex["en"] for ex in examples["translation"]]
  model_inputs = tokenizer(
      inputs, text_target=targets, max_length=max_length, truncation=True
  )

  return model_inputs

In [ ]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names
)

Map:   0%|          | 0/38503 [00:00<?, ? examples/s]

Map:   0%|          | 0/4279 [00:00<?, ? examples/s]

## **Fine-tuning the model with the Trainer API**

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

### **Data collation**

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[ 8804,  2075,  6010,   851,     0, 53738, 53738, 53738, 53738, 53738,
         53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738,
         53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738,
         53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738,
         53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738, 53738,
         53738, 53738, 53738, 53738],
        [39539,   123,   131,  3827,    43,    75,   123,   669,   791,   324,
             2,  2937,  2399,  1328,     3,   276,   848,   606,   357,    17,
            13,   941,  1901,    28,   139,  2317,    52,  1076,   653,  1076,
          2029,   167, 14938,   139,     3, 16320,   123,  3827,    43,    34,
          2010,  1009,   941,  1901,    28,   139,  2317,    52,  1076,   653,
         14938,   139,     2,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0

### **Metrics**

In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


In [ ]:
import evaluate

metric = evaluate.load("sacrebleu")

In [ ]:
predictions = [
    "Thanh tiêu đề không chọn"
]
references = [
    [
        "Thanh tiêu đề này không được chọn"
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 30.28512683288236,
 'counts': [5, 2, 1, 0],
 'totals': [5, 4, 3, 2],
 'precisions': [100.0, 50.0, 33.333333333333336, 25.0],
 'bp': 0.6703200460356393,
 'sys_len': 5,
 'ref_len': 7}

In [ ]:
import numpy as np

def compute_metrics(eval_preds):
  preds, labels = eval_preds

  if isinstance(preds, tuple):
    preds = preds[0]

  decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

  labels = np.where(labels != 100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  decoded_preds = [pred.strip() for pred in decoded_preds]
  decoded_labels = [label.strip() for label in decoded_labels]

  result = metric.compute(predictions=decoded_preds, references=decoded_labels)
  return {"bleu": result["score"]}

### **Fine-tuning the model**

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"Finetuned-kde4-vi-to-en",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True
)

Cuối cùng, ta chỉ cần truyền mọi thứ cho Seq2SeqTrainer:

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-588655935.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.evaluate(max_length=max_length)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vo-minhbao1705 (vo-minhbao1705-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'eval_loss': 1.754225254058838,
 'eval_model_preparation_time': 0.0184,
 'eval_bleu': 45.20647379801491,
 'eval_runtime': 226.802,
 'eval_samples_per_second': 18.867,
 'eval_steps_per_second': 0.295}

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,1.834100
1000,1.655400
1500,1.554600


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4034: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[53738]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1806, training_loss=1.6540425449510745, metrics={'train_runtime': 496.9789, 'train_samples_per_second': 232.422, 'train_steps_per_second': 3.634, 'total_flos': 2378724803149824.0, 'train_loss': 1.6540425449510745, 'epoch': 3.0})

In [29]:
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...i-to-en/training_args.bin: 100%|##########| 5.97kB / 5.97kB            

  ...873357.2ce9063a608b.838.0: 100%|##########| 7.17kB / 7.17kB            

  ...-kde4-vi-to-en/target.spm: 100%|##########|  809kB /  809kB            

  ...-kde4-vi-to-en/source.spm: 100%|##########|  756kB /  756kB            

  ...i-to-en/model.safetensors:   3%|2         | 8.35MB /  287MB            

  ...874170.2ce9063a608b.838.1: 100%|##########|   473B /   473B            

CommitInfo(commit_url='https://huggingface.co/Ara113/Finetuned-kde4-vi-to-en/commit/64aae0271240c191025fa4646a10d3e6c39b07d8', commit_message='End of training', commit_description='', oid='64aae0271240c191025fa4646a10d3e6c39b07d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ara113/Finetuned-kde4-vi-to-en', endpoint='https://huggingface.co', repo_type='model', repo_id='Ara113/Finetuned-kde4-vi-to-en'), pr_revision=None, pr_num=None)

In [28]:
trainer.evaluate(max_length=max_length)

{'eval_loss': 1.5147181749343872,
 'eval_model_preparation_time': 0.0184,
 'eval_bleu': 47.651710757229324,
 'eval_runtime': 230.3532,
 'eval_samples_per_second': 18.576,
 'eval_steps_per_second': 0.291,
 'epoch': 3.0}

In [30]:
from transformers import pipeline

# Thay nó với checkpoint của bạn
model_checkpoint = "Ara113/Finetuned-kde4-vi-to-en"
translator = pipeline("translation", model=model_checkpoint)
translator("Xin chào Việt Nam")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/287M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


[{'translation_text': 'Hello, Vietnam.'}]

In [31]:
translator("Chân thành cảm ơn mọi người đã ghé xem livestream của mình")

[{'translation_text': 'Thank you so much for visiting your livestream.'}]